# Run17 Test Training (Local)
Thin orchestration notebook for the canonical `run17_test` 100k-timestep comparison path.
Use this notebook for short A/B tests such as Dirichlet alpha-generator comparisons before returning to the main Run17 run.
Core logic lives in `src/config.py`, `src/notebook_helpers/tcn_phase1.py`, and the agent/environment source files.


## 1) Colab Setup
Clone/sync the repo, clean previous outputs, install requirements, and verify GPU availability.


In [1]:
import shutil
from pathlib import Path
repo = Path("/content/tcn_tape_vectorized_version_clean")
if repo.exists():
    shutil.rmtree(repo, ignore_errors=True)
print("removed", repo)

removed /content/tcn_tape_vectorized_version_clean


In [2]:
import gc
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

# GitHub-native setup knobs.
GIT_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
GIT_BRANCH = "feature/run17-film-drive-org-20260318"    # e.g. 'feature/run17-film-drive-org-20260318'
CLONE_IF_MISSING = True
CLONE_PARENT_DIR = Path('/content')
CLONE_DIR_NAME = 'tcn_tape_vectorized_version_clean'

# Notebook/runtime knobs.
TRAIN_BRANCH = None  # Optional post-clone/post-open checkout override.
INSTALL_REQUIREMENTS = True
AUTO_INSTALL_MISSING_REQUIREMENTS = True
RESET_OUTPUT_DIRS = True


def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True)


CRITICAL_RUNTIME_MODULES = {
    'pandas_ta_classic': 'pandas-ta-classic>=0.3.59',
    'fredapi': 'fredapi>=0.5.1',
    'yfinance': 'yfinance>=0.2.38',
}


def missing_runtime_requirements() -> list[str]:
    missing = []
    for module_name, requirement in CRITICAL_RUNTIME_MODULES.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(requirement)
    return missing


def normalize_github_url(url: str | None) -> str | None:
    if not url:
        return url
    url = str(url).strip()
    if url.startswith('git@github.com:'):
        repo = url[len('git@github.com:'):]
        if repo.endswith('.git'):
            repo = repo[:-4]
        return f'https://github.com/{repo}.git'
    return url


def find_repo_root() -> Path | None:
    candidate_roots = []
    seen = set()
    for p in [Path.cwd(), *Path.cwd().parents]:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            candidate_roots.append(rp)
    for p in [
        CLONE_PARENT_DIR / CLONE_DIR_NAME,
        Path('/content/repo'),
        Path('/content/project'),
        Path('C:/Users/Owner/tcn_tape_vectorized_version_clean'),
        Path('/mnt/c/Users/Owner/tcn_tape_vectorized_version_clean'),
    ]:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            candidate_roots.append(rp)

    for p in candidate_roots:
        if (p / '.git').exists() and (p / 'src').exists() and (p / 'tcn_architecture_analysis.ipynb').exists():
            return p
    return None


TRAIN_REPO_DIR = find_repo_root()
GIT_REPO_URL = normalize_github_url(GIT_REPO_URL)

if TRAIN_REPO_DIR is None and CLONE_IF_MISSING:
    if not GIT_REPO_URL:
        raise FileNotFoundError(
            'Repo root not found and GIT_REPO_URL is not set. '
            'Set GIT_REPO_URL to your GitHub clone URL for a fresh runtime.'
        )
    CLONE_PARENT_DIR.mkdir(parents=True, exist_ok=True)
    clone_target = CLONE_PARENT_DIR / CLONE_DIR_NAME
    if clone_target.exists() and not (clone_target / '.git').exists():
        print(f'[WARN] Removing partial non-git clone target: {clone_target}')
        shutil.rmtree(clone_target, ignore_errors=True)
    if not clone_target.exists():
        try:
            run(['git', 'clone', GIT_REPO_URL, str(clone_target)])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                'git clone failed. Use an HTTPS GitHub URL in GIT_REPO_URL. '                f'Normalized URL: {GIT_REPO_URL}'
            ) from exc
    TRAIN_REPO_DIR = clone_target.resolve()

if TRAIN_REPO_DIR is None:
    attempted = '\n'.join([
        f' - {Path.cwd().resolve()}',
        *[f' - {p.resolve()}' for p in Path.cwd().parents],
        f' - {(CLONE_PARENT_DIR / CLONE_DIR_NAME).resolve()}',
        ' - /content/repo',
        ' - /content/project',
        ' - C:/Users/Owner/tcn_tape_vectorized_version_clean',
        ' - /mnt/c/Users/Owner/tcn_tape_vectorized_version_clean',
    ])
    raise FileNotFoundError('Repo root not found. Tried:\n' + attempted)

REQUESTED_BRANCH = TRAIN_BRANCH or GIT_BRANCH
if REQUESTED_BRANCH:
    run(['git', '-C', str(TRAIN_REPO_DIR), 'fetch', 'origin'])
    run(['git', '-C', str(TRAIN_REPO_DIR), 'checkout', REQUESTED_BRANCH])
    # In Colab, prefer the exact remote branch tip over any stale local branch state.
    try:
        run(['git', '-C', str(TRAIN_REPO_DIR), 'reset', '--hard', f'origin/{REQUESTED_BRANCH}'])
    except subprocess.CalledProcessError:
        print(f'[WARN] No origin tracking ref found for {REQUESTED_BRANCH}; using local checkout only.')

if RESET_OUTPUT_DIRS:
    purge_paths = [
        TRAIN_REPO_DIR / 'tcn_fusion_results',
        TRAIN_REPO_DIR / 'tcn_results',
        TRAIN_REPO_DIR / 'tcn_att_results',
        TRAIN_REPO_DIR / 'results',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data' / 'daily_ohlcv_assets.csv',
        TRAIN_REPO_DIR / 'data' / 'processed_daily_macro_features.csv',
    ]
    for path in purge_paths:
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
        elif path.exists():
            path.unlink()

for cache_dir in TRAIN_REPO_DIR.rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)

for ckpt_dir in TRAIN_REPO_DIR.rglob('.ipynb_checkpoints'):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules):
    if mod == 'src' or mod.startswith('src.'):
        del sys.modules[mod]

gc.collect()

os.chdir(TRAIN_REPO_DIR)
if str(TRAIN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_REPO_DIR))

requirements_file = TRAIN_REPO_DIR / 'requirements.txt'
missing_requirements = missing_runtime_requirements()
should_install = INSTALL_REQUIREMENTS or (AUTO_INSTALL_MISSING_REQUIREMENTS and bool(missing_requirements))
if should_install:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
    if INSTALL_REQUIREMENTS:
        run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)])
    elif missing_requirements:
        run([sys.executable, '-m', 'pip', 'install', *missing_requirements])

print('[OK] Repo ready:', TRAIN_REPO_DIR)
print('[OK] Normalized GIT_REPO_URL:', GIT_REPO_URL)
run(['git', '-C', str(TRAIN_REPO_DIR), 'rev-parse', '--abbrev-ref', 'HEAD'])
run(['git', '-C', str(TRAIN_REPO_DIR), 'rev-parse', 'HEAD'])
print('[OK] Requirements install requested:', INSTALL_REQUIREMENTS)
print('[OK] Missing critical requirements before install:', missing_requirements)
print('[OK] Dependency install executed:', should_install)
print('[OK] GitHub clone-if-missing:', CLONE_IF_MISSING)
print('[OK] Requested git branch:', GIT_BRANCH)


+ git clone https://github.com/Dave-DKings/tcn_tape_vectorized_version.git /content/tcn_tape_vectorized_version_clean
+ git -C /content/tcn_tape_vectorized_version_clean fetch origin
+ git -C /content/tcn_tape_vectorized_version_clean checkout feature/run17-film-drive-org-20260318
+ git -C /content/tcn_tape_vectorized_version_clean reset --hard origin/feature/run17-film-drive-org-20260318
+ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
+ /usr/bin/python3 -m pip install -r /content/tcn_tape_vectorized_version_clean/requirements.txt
[OK] Repo ready: /content/tcn_tape_vectorized_version_clean
[OK] Normalized GIT_REPO_URL: https://github.com/Dave-DKings/tcn_tape_vectorized_version.git
+ git -C /content/tcn_tape_vectorized_version_clean rev-parse --abbrev-ref HEAD
+ git -C /content/tcn_tape_vectorized_version_clean rev-parse HEAD
[OK] Requirements install requested: True
[OK] Missing critical requirements before install: []
[OK] Dependency install executed: True
[OK] GitHub

In [3]:
import tensorflow as tf
import subprocess

REQUIRE_GPU = False  # set True if you want hard fail without GPU

try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    if smi.returncode == 0:
        print("nvidia-smi:", [line.strip() for line in smi.stdout.splitlines() if line.strip()])
    else:
        print("nvidia-smi: not available")
except Exception:
    print("nvidia-smi: not available")

gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)

if not gpus:
    msg = 'No GPU visible to TensorFlow; running on CPU (slower).'
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('[WARN]', msg)
else:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

tf.keras.mixed_precision.set_global_policy('float32')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
print('TF build CUDA:', tf.test.is_built_with_cuda())


nvidia-smi: ['NVIDIA A100-SXM4-80GB']
TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
TF build CUDA: True


## 2) Imports
Import the canonical source helpers and training entrypoints.


In [4]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import build_run17_test_config, assert_run17_test_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import (
    _get_results_root_for_architecture,
    prepare_phase1_dataset,
    run_experiment6_tape,
)

RUN_ID = 'run17_test'
TRAIN_RANDOM_SEED = 42
ANALYSIS_END_DATE = None

RUN_CONFIG_LINEAGE = 'run17_test'
ENABLE_DIAGNOSTIC_OVERRIDE = True
DIAGNOSTIC_ASSET_TICKERS = ['NVDA', 'MSFT', 'JPM', 'XOM', 'GLD']
DIAGNOSTIC_TCN_FILTERS = [64, 96, 128]
DIAGNOSTIC_TCN_DILATIONS = [1, 2, 4]
DIAGNOSTIC_TCN_KERNEL_SIZE = 5
DIAGNOSTIC_DIRICHLET_ALPHA_ACTIVATION = 'cross_softplus'
DIAGNOSTIC_DIRICHLET_SOFTPLUS_ALPHA_FLOOR = 0.5
DIAGNOSTIC_DIRICHLET_SOFTPLUS_ALPHA_SCALE = 2.0
DIAGNOSTIC_DIRICHLET_CROSS_SECTIONAL_STANDARDIZE = True

In [5]:
import logging

QUIET_SRC_INFO_LOGS = True

if QUIET_SRC_INFO_LOGS:
    # Suppress noisy module INFO logs like: "... - src.environment_tape_rl - INFO ..."
    for logger_name in [
        "src",
        "src.environment_tape_rl",
        "src.notebook_helpers.tcn_phase1",
        "src.agents.ppo_agent_tf",
    ]:
        logging.getLogger(logger_name).setLevel(logging.WARNING)
    print("[OK] Suppressed src INFO logs (level=WARNING).")
else:
    print("[INFO] Src INFO logs left enabled.")


[OK] Suppressed src INFO logs (level=WARNING).


## 3) Build Canonical Run17 Config and Dataset
Create the source-backed Run17 config (20-asset expanded universe), assert no drift, and prepare the dataset once.

In [6]:
train_config = build_run17_test_config('phase1', analysis_end_date=ANALYSIS_END_DATE)
assert_run17_test_config(train_config)

if ENABLE_DIAGNOSTIC_OVERRIDE:
    diagnostic_num_assets = len(DIAGNOSTIC_ASSET_TICKERS)
    train_config['ASSET_TICKERS'] = list(DIAGNOSTIC_ASSET_TICKERS)
    train_config['NUM_ASSETS'] = diagnostic_num_assets
    train_config['EQUAL_WEIGHT_CASH_ALLOCATION'] = 1.0 / float(diagnostic_num_assets + 1)

    feature_params = train_config.setdefault('feature_params', {})
    dyn_cov = feature_params.get('dynamic_covariance', {}) if isinstance(feature_params.get('dynamic_covariance', {}), dict) else {}
    dyn_cov['num_eigenvalues'] = min(2, diagnostic_num_assets)
    feature_params['dynamic_covariance'] = dyn_cov

    ap = train_config['agent_params']
    ap['num_assets'] = diagnostic_num_assets
    ap['tcn_filters'] = list(DIAGNOSTIC_TCN_FILTERS)
    ap['tcn_dilations'] = list(DIAGNOSTIC_TCN_DILATIONS)
    ap['tcn_kernel_size'] = int(DIAGNOSTIC_TCN_KERNEL_SIZE)
    ap['dirichlet_alpha_activation'] = str(DIAGNOSTIC_DIRICHLET_ALPHA_ACTIVATION)
    ap['dirichlet_softplus_alpha_floor'] = float(DIAGNOSTIC_DIRICHLET_SOFTPLUS_ALPHA_FLOOR)
    ap['dirichlet_softplus_alpha_scale'] = float(DIAGNOSTIC_DIRICHLET_SOFTPLUS_ALPHA_SCALE)
    ap['dirichlet_cross_sectional_standardize'] = bool(DIAGNOSTIC_DIRICHLET_CROSS_SECTIONAL_STANDARDIZE)
    ap.pop('state_layout', None)

    env = train_config['environment_params']
    env['stock_dim'] = diagnostic_num_assets
    env['num_assets'] = diagnostic_num_assets

    print('[Run17_test] Diagnostic override active')
    print('  config_lineage =', RUN_CONFIG_LINEAGE)
    print('  diagnostic_assets =', DIAGNOSTIC_ASSET_TICKERS)
    print('  diagnostic_tcn_filters =', DIAGNOSTIC_TCN_FILTERS)
    print('  diagnostic_tcn_dilations =', DIAGNOSTIC_TCN_DILATIONS)
    print('  diagnostic_tcn_kernel_size =', DIAGNOSTIC_TCN_KERNEL_SIZE)
    print('  diagnostic_dirichlet_alpha_activation =', DIAGNOSTIC_DIRICHLET_ALPHA_ACTIVATION)
    print('  diagnostic_dirichlet_softplus_alpha_floor =', DIAGNOSTIC_DIRICHLET_SOFTPLUS_ALPHA_FLOOR)
    print('  diagnostic_dirichlet_softplus_alpha_scale =', DIAGNOSTIC_DIRICHLET_SOFTPLUS_ALPHA_SCALE)
    print('  diagnostic_dirichlet_cross_sectional_standardize =', DIAGNOSTIC_DIRICHLET_CROSS_SECTIONAL_STANDARDIZE)


tp = train_config['training_params']
ap = train_config['agent_params']
ppo = ap['ppo_params']
env = train_config['environment_params']

print('[Run17_test] Config ready for training')
print('  num_assets =', train_config['NUM_ASSETS'])
print('  tickers =', train_config['ASSET_TICKERS'])
print('  analysis_start_date =', train_config['ANALYSIS_START_DATE'])
print('  split_date =', train_config['TRAIN_TEST_SPLIT_DATE'])
print('  architecture =', ap['actor_critic_type'])
print('  tcn_filters =', ap.get('tcn_filters'))
print('  tcn_dilations =', ap.get('tcn_dilations'))
print('  tcn_kernel_size =', ap.get('tcn_kernel_size'))
print('  dirichlet_alpha_activation =', ap.get('dirichlet_alpha_activation'))
print('  dirichlet_softplus_alpha_floor =', ap.get('dirichlet_softplus_alpha_floor'))
print('  dirichlet_softplus_alpha_scale =', ap.get('dirichlet_softplus_alpha_scale'))
print('  dirichlet_cross_sectional_standardize =', ap.get('dirichlet_cross_sectional_standardize'))
print('  regime_conditioning =', ap['regime_conditioning_enabled'])
print('  distributional_critic =', ap['distributional_critic_enabled'])
print('  cvar_advantage_weight =', ppo['cvar_advantage_weight'])
print('  lagrangian =', {
    'enabled': ppo['lagrangian_cvar_enabled'],
    'threshold': ppo['lagrangian_cvar_threshold'],
    'lr': ppo['lagrangian_cvar_lr'],
    'lambda_max': ppo['lagrangian_cvar_lambda_max'],
    'penalty_scale': ppo['lagrangian_cvar_penalty_scale'],
})
print('  drawdown =', {
    'target': env['drawdown_constraint']['target'],
    'tolerance': env['drawdown_constraint']['tolerance'],
    'penalty_coef': env['drawdown_constraint']['penalty_coef'],
    'lambda_carry_decay': env['drawdown_constraint']['lambda_carry_decay'],
})
print('  dispersion =', {
    'hhi_coef': ppo['alpha_diversity_coef'],
    'dispersion_coef': ppo['alpha_dispersion_coef'],
    'dispersion_target_std': ppo['alpha_dispersion_target_std'],
})
print('  beta_curriculum =', tp['action_execution_beta_curriculum'])
print('  turnover_curriculum =', tp['turnover_penalty_curriculum'])
print('  reward_component_schedule =', tp['reward_component_schedule'])
print('  deterministic_validation =', tp['deterministic_validation_checkpointing_enabled'])
print('  training_early_stop =', tp.get('training_early_stop_enabled', False))

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(TRAIN_REPO_DIR / 'data_exports'),
)

actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Actuarial_')]
if actuarial_cols:
    raise RuntimeError(f'Actuarial columns should be absent for Run17_test: {actuarial_cols}')

alpha_ret_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('AlphaRet_')]
expected_alpha_cols = {'AlphaRet_1d', 'AlphaRet_5d', 'AlphaRet_20d', 'AlphaRet_5d_Z', 'AlphaRet_20d_Z'}
missing_alpha_cols = sorted(list(expected_alpha_cols - set(alpha_ret_cols)))
if missing_alpha_cols:
    raise RuntimeError(f'Missing expected alpha-return columns: {missing_alpha_cols}')

fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Fundamental_')]
if fundamental_cols:
    raise RuntimeError(f'Fundamental columns should be absent: {fundamental_cols}')

observed_tickers = sorted(set(train_phase1_data.master_df['Ticker'].astype(str))) if 'Ticker' in train_phase1_data.master_df.columns else []
missing_tickers = sorted(set(DIAGNOSTIC_ASSET_TICKERS) - set(observed_tickers)) if ENABLE_DIAGNOSTIC_OVERRIDE else []
if missing_tickers:
    raise RuntimeError(f'Missing diagnostic asset tickers in prepared dataset: {missing_tickers}')

print('[OK] Train shape:', train_phase1_data.train_df.shape)
print('[OK] Test shape:', train_phase1_data.test_df.shape)
print('[OK] Observed tickers:', observed_tickers)
print('[OK] Actuarial feature check passed: none present (disabled)')
print('[OK] Alpha-return feature check passed:', sorted(alpha_ret_cols)[:10])
print('[OK] Fundamental feature check passed: none present')






[Run17_test] Diagnostic override active
  config_lineage = run17_test
  diagnostic_assets = ['NVDA', 'MSFT', 'JPM', 'XOM', 'GLD']
  diagnostic_tcn_filters = [64, 96, 128]
  diagnostic_tcn_dilations = [1, 2, 4]
  diagnostic_tcn_kernel_size = 5
  diagnostic_dirichlet_alpha_activation = cross_softplus
  diagnostic_dirichlet_softplus_alpha_floor = 0.5
  diagnostic_dirichlet_softplus_alpha_scale = 2.0
  diagnostic_dirichlet_cross_sectional_standardize = True
[Run17_test] Config ready for training
  num_assets = 5
  tickers = ['NVDA', 'MSFT', 'JPM', 'XOM', 'GLD']
  analysis_start_date = 2009-01-01
  split_date = 2019-12-31
  architecture = TCN_FUSION
  tcn_filters = [64, 96, 128]
  tcn_dilations = [1, 2, 4]
  tcn_kernel_size = 5
  dirichlet_alpha_activation = cross_softplus
  dirichlet_softplus_alpha_floor = 0.5
  dirichlet_softplus_alpha_scale = 2.0
  dirichlet_cross_sectional_standardize = True
  regime_conditioning = True
  distributional_critic = True
  cvar_advantage_weight = 0.1
  lagr

## 4) Run Training
Launch the canonical Run17 training path.


In [7]:
RUN_TRAINING = True

if RUN_TRAINING:
    training_params = train_config['training_params']
    print('[START] Starting training')
    print('Architecture:', train_config['agent_params'].get('actor_critic_type'))
    print('max_total_timesteps:', training_params['max_total_timesteps'])
    print('num_parallel_envs:', training_params.get('num_parallel_envs', 1))

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config['agent_params'].get('actor_critic_type'),
        timesteps_per_update=training_params.get('timesteps_per_ppo_update', 1008),
        max_total_timesteps=training_params['max_total_timesteps'],
    )

    print('[OK] Training complete')
    print('checkpoint_prefix:', train_experiment6.checkpoint_path)
else:
    print('[SKIP] RUN_TRAINING=False')

[START] Starting training
Architecture: TCN_FUSION
max_total_timesteps: 100000
num_parallel_envs: 8

EXPERIMENT 6: TCN_FUSION Enhanced + TAPE Three-Component
Architecture: TCN + Fusion
Results root: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results
Working dir: /content/tcn_tape_vectorized_version_clean
Covariance Features: Yes
🎯 REWARD SYSTEM: TAPE (Three-Component v3)
   Profile: BalancedGrowth
   Daily: Base + DSR/PBRS + Turnover_Proximity
   Terminal: mode=signed | baseline=0.20 | scalar=10.0 (clipped ±10.0)
   Gate A: enabled (Sharpe <= 0.00 or MDD >= 25.0% -> force non-positive terminal bonus)
   Neutral Band: enabled (±0.020 around baseline)
   [CYCLE] Profile Manager: disabled (static profile only)
[RAND] Experiment Seed: 6042 (Base: 42, Offset: 6000)
[OK] Features: Enhanced (includes 2 covariance eigenvalues)
   Eigenvalues: ['Covariance_Eigenvalue_0', 'Covariance_Eigenvalue_1']
   Train shape: (13840, 67)
   Test shape: (7115, 67)
   ℹ️ Actuarial features disabled

## 5) Inspect Latest Training Logs
Load the latest training CSVs and inspect the current run.


In [8]:
TRAIN_RESULTS_ROOT = _get_results_root_for_architecture(
    architecture=train_config['agent_params'].get('actor_critic_type', 'TCN_FUSION'),
    use_attention=bool(train_config['agent_params'].get('use_attention', False)),
    use_fusion=bool(train_config['agent_params'].get('use_fusion', False)),
    project_root=TRAIN_REPO_DIR,
)
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / 'logs'

print('Results root:', TRAIN_RESULTS_ROOT)
print('Logs dir:', TRAIN_LOGS_DIR)

episodes_files = sorted(TRAIN_LOGS_DIR.glob('*episodes*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f'No episodes CSV found in {TRAIN_LOGS_DIR}')
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print('Episodes file:', train_episodes_path)
    print('Rows:', len(train_episodes_df))
    display(train_episodes_df.tail(20))

step_diag_files = sorted(TRAIN_LOGS_DIR.glob('*step_diagnostics*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if step_diag_files:
    step_diag_path = step_diag_files[0]
    step_diag_df = pd.read_csv(step_diag_path)
    print('Step diagnostics file:', step_diag_path)
    print('Rows:', len(step_diag_df))
    display(step_diag_df.tail(20))


Results root: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results
Logs dir: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/logs
Episodes file: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260319_152902_episodes.csv
Rows: 70


,update,timestep,episode,elapsed_time,episode_return_pct,episode_sharpe,episode_sortino,episode_max_dd,episode_volatility,episode_win_rate,...,alpha_mean,alpha_cap_hit_frac,mixture_balance_loss,mixture_separation_loss,mixture_component_dispersion_loss,mixture_gating_entropy,mixture_component_usage,ratio_mean,ratio_std,drawdown_lambda_peak
50,51,62496,56,5753.628640,-23.959544,-1.012658,-1.387710,28.881701,0.079540,45.462963,...,1.610545,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",0.997387,0.634457,5.000000
51,52,64512,56,5906.680694,-18.422895,-0.653329,-0.960528,28.881701,0.083915,46.096096,...,1.606858,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",1.002389,0.603147,5.000000
52,53,66528,64,6061.557179,-35.965080,-1.167498,-1.555043,39.077714,0.078498,46.897932,...,1.603342,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",1.003377,0.634659,5.000000
53,54,68544,64,6214.403445,-22.839379,-1.779710,-2.278125,25.296329,0.116537,44.345238,...,1.597731,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",0.997558,0.597943,5.000000
54,55,70560,64,6367.375850,-28.818295,-1.537118,-1.982398,30.406675,0.104107,43.197279,...,1.593496,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",1.016542,0.635512,5.000000
55,56,72576,64,6520.277062,-32.251622,-1.410709,-1.839963,34.313875,0.093707,43.214286,...,1.589797,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",0.998553,0.587127,5.000000
56,57,74592,64,6673.296519,-40.491958,-1.516438,-1.996119,41.297915,0.089394,42.307692,...,1.586107,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",0.995639,0.613145,5.000000
57,58,76608,64,6826.359382,-46.217586,-1.461979,-1.952362,50.024451,0.090285,42.559524,...,1.581739,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",0.987741,0.569824,5.000000
58,59,78624,72,6979.655731,-33.434425,-1.059855,-1.421912,38.040942,0.080199,46.631087,...,1.578129,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",1.001735,0.616816,5.000000
59,60,80640,72,7132.037965,-6.509446,-1.122541,-1.459153,9.554921,0.059485,47.413793,...,1.572907,0.0,0.0,0.0,0.0,0.0,"[0.0, 0.0, 0.0]",0.987504,0.553515,5.000000


Step diagnostics file: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/logs/Exp6_TCN_FUSION_Enhanced_TAPE_training_20260319_152902_step_diagnostics.csv
Rows: 100000


,update,timestep,episode,episode_step,date,elapsed_time,reward_total,portfolio_return_pct_points,portfolio_value,prev_portfolio_value,...,turnover_penalty_contrib,transaction_cost_dollars,tx_cost_contrib_reward_pts,action_realization_l1,action_realization_penalty,cash_weight_raw,cash_weight_projected,cash_weight_final,cash_weight_forced_gap,drawdown_penalty
99980,70,99981,85,409,2016-07-28 00:00:00,8610.409375,0.314355,-0.110352,89812.864188,89912.084203,...,-0.019093,32.947039,-0.036644,0.000000,0.0,0.734650,0.734650,0.734650,0.0,0.000000
99981,70,99982,85,64,2010-08-25 00:00:00,8610.413731,0.264142,0.142023,92163.609140,92032.901595,...,-0.591455,80.023480,-0.086951,0.000000,0.0,0.688235,0.688235,0.688235,0.0,0.000000
99982,70,99983,85,735,2017-09-18 00:00:00,8610.418422,-0.701364,-0.192402,99365.486863,99557.036576,...,-0.410984,70.560895,-0.070875,0.274725,0.0,0.423101,0.423101,0.423101,0.0,0.000000
99983,70,99984,85,273,2018-08-01 00:00:00,8610.422862,-0.837982,-0.102289,102292.417591,102397.158588,...,-0.328864,65.279198,-0.063751,0.064100,0.0,0.466300,0.466300,0.466300,0.0,0.000000
99984,70,99985,85,676,2012-10-24 00:00:00,8610.854375,-1.175412,-0.375692,88334.286061,88667.402146,...,-0.701829,85.245270,-0.096140,0.033504,0.0,0.397083,0.397083,0.397083,0.0,0.048751
99985,70,99986,85,1009,2015-11-19 00:00:00,8610.859342,-2.316786,0.032654,70296.898467,70273.951251,...,-0.269061,41.177888,-0.058596,0.000000,0.0,0.454521,0.454521,0.454521,0.0,1.074708
99986,70,99987,85,772,2018-01-09 00:00:00,8610.864135,-0.729286,-0.140090,93584.240783,93715.526609,...,-0.178132,47.365023,-0.050541,0.000000,0.0,0.539006,0.539006,0.539006,0.0,0.000000
99987,70,99988,85,305,2014-08-04 00:00:00,8610.868762,-0.663451,0.271176,85535.106926,85303.783778,...,-0.722527,84.096112,-0.098584,0.000000,0.0,0.330224,0.330224,0.330224,0.0,0.000000
99988,70,99989,85,410,2016-07-29 00:00:00,8610.873399,0.759806,0.238759,90027.300626,89812.864188,...,-0.405199,63.473366,-0.070673,0.183822,0.0,0.473833,0.473833,0.473833,0.0,0.000000
99989,70,99990,85,65,2010-08-26 00:00:00,8610.877821,-1.374362,-0.494871,91707.517962,92163.609140,...,-0.209190,48.909949,-0.053069,0.000000,0.0,0.548450,0.548450,0.548450,0.0,0.000000


## 6) Export Artifacts (Optional)
Zip the latest results and save them into a dedicated Google Drive folder for this run.


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import shutil
import subprocess

EXPORT_RESULTS_ZIP = True
SAVE_TO_DRIVE = True
DRIVE_EXPORT_ROOT = Path('/content/drive/MyDrive/tcn_tape_vectorized_runs')
DRIVE_EXPORT_DIR = DRIVE_EXPORT_ROOT / RUN_ID
LOCAL_EXPORT_PATH = TRAIN_REPO_DIR / f"tcn_tape_vectorized_{RUN_ID}.zip"

if 'TRAIN_RESULTS_ROOT' not in globals():
    TRAIN_RESULTS_ROOT = _get_results_root_for_architecture(
        architecture=train_config['agent_params'].get('actor_critic_type', 'TCN_FUSION'),
        use_attention=bool(train_config['agent_params'].get('use_attention', False)),
        use_fusion=bool(train_config['agent_params'].get('use_fusion', False)),
        project_root=TRAIN_REPO_DIR,
    )

if EXPORT_RESULTS_ZIP:
    include_paths = [
        TRAIN_RESULTS_ROOT,
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data_exports',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
    ]
    existing = []
    seen = set()
    for path in include_paths:
        resolved = path.resolve()
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            existing.append(resolved)

    if existing:
        if LOCAL_EXPORT_PATH.exists():
            LOCAL_EXPORT_PATH.unlink()
        relative_items = [str(path.relative_to(TRAIN_REPO_DIR)) for path in existing]
        subprocess.run(
            [
                'bash',
                '-lc',
                'cd "{}" && zip -qr "{}" {}'.format(
                    TRAIN_REPO_DIR,
                    LOCAL_EXPORT_PATH,
                    ' '.join(f'"{item}"' for item in relative_items),
                ),
            ],
            check=True,
        )
        print('[OK] Created local zip:', LOCAL_EXPORT_PATH)

        if SAVE_TO_DRIVE:
            if not Path('/content/drive/MyDrive').exists():
                raise FileNotFoundError('Google Drive is not mounted at /content/drive/MyDrive')
            DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
            drive_zip_path = DRIVE_EXPORT_DIR / LOCAL_EXPORT_PATH.name
            shutil.copy2(LOCAL_EXPORT_PATH, drive_zip_path)
            print('[OK] Copied zip to Drive:', drive_zip_path)
            print('[OK] Drive run folder:', DRIVE_EXPORT_DIR)
    else:
        print('[WARN] Nothing to export.')
else:
    print('[SKIP] EXPORT_RESULTS_ZIP=False')
    print('Drive export folder would be:', DRIVE_EXPORT_DIR)


[OK] Created local zip: /content/tcn_tape_vectorized_version_clean/tcn_tape_vectorized_run17_test.zip
[OK] Copied zip to Drive: /content/drive/MyDrive/tcn_tape_vectorized_runs/run17_test/tcn_tape_vectorized_run17_test.zip
[OK] Drive run folder: /content/drive/MyDrive/tcn_tape_vectorized_runs/run17_test
